# Final two experiments: V24 dictionary-all and dictionary + neural fallback

**These variants retain dictionary lookup. They do not meet the organizers’ requested general, always-on neural method.** A searches every input, but can leave unmatched clips unchanged. B routes by accepted dictionary change, so it is a hybrid experiment rather than a natural/synthetic oracle. No claim of organizer acceptance, novelty, or score improvement is made.

This notebook faithfully reuses the uploaded V24 implementation and its frozen artifacts. It does not establish that V24 reproduces your best V23 score. Full-bank refitting is cached; no Phase-2 waveform is used to build noise templates. Existing V24 outputs and proposals can be reused after identity checks. V36 Demucs is used only on unmatched clips in variant B. Both packages receive fresh/content-cached SraVaani transcripts of their exact final PCM audio.

Default neural fallback is V36 teacher epoch 1, α=0.95, the Demucs checkpoint used with α=0.95 in your reported V36 submission. The reported V36 server score was SI-SDR 9.3877 dB, combined 8.8878. Hybrid performance is unmeasured. Edit the explicit checkpoint if desired; automatic guesses are avoided. No new model training occurs. Run cells in order in your existing GPU Colab environment.


## 1. Paths and environment

In [ ]:
# Original V24 paths; existing output/cache locations are retained.
from pathlib import Path
import os, sys

os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    print('Outside Colab: edit paths below.')

WORK = Path('/content/drive/MyDrive/Datathon/work')
V5_DIR = WORK / 'v5_synthetic_residual_evidence'

# Exact scored V6 selection run from Phase 1.
V6_ROOT = Path('/content/drive/MyDrive/Datathon/track2_v6_direct_reconstruction')
V6_RUN_KEY = '23523d3d7e5ed3ea'
V6_RUN = V6_ROOT / ('run_' + V6_RUN_KEY)

# Released labelled development data used to build the final deployment bank.
VAL_CACHE = WORK / 'official_val_posteriors_epoch_2.pkl'
VAL_ROOT = None
VAL_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Datathon/validation'),
    Path('/content/drive/MyDrive/PHD/Datathon/validation'),
    Path('/content/drive/MyDrive/validation'),
]

# NEW Phase-2 input from Codabench.
PHASE2_INPUT_SOURCE = Path('/content/drive/MyDrive/PHD/Datathon/input_data.zip')
EXPECTED_PHASE2_CLIPS = 3178

OUT_ROOT = Path('/content/drive/MyDrive/Datathon/track2_phase2_v24_fullbank')
LOCAL_ROOT = Path('/content/track2_phase2_v24_fullbank')
OUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ASR = True
ASR_REPO = 'ARTPARK-IISc/SraVaani-1.0'
ASR_BATCH_SIZE = 8

# Reuse exact immutable revision if available.
OLD_ASR_PIN = V6_ROOT / 'sravaani_revision_pin.json'

# Keep final WAVs local to avoid another ~1 GB Drive copy.
# Only the final ZIP + audits are copied to Drive.
PERSIST_FINAL_WAVS = False

print('V24 Phase-2 full released-bank refit')
print('Phase-2 input:', PHASE2_INPUT_SOURCE)
print('Expected clips:', EXPECTED_PHASE2_CLIPS)
print('V5 artifacts:', V5_DIR)
print('Frozen V6 run:', V6_RUN)

os.environ['USE_TF']='0';os.environ['USE_FLAX']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
import gc,importlib.util,subprocess
EXPERIMENT_ROOT=Path('/content/drive/MyDrive/PHD/Datathon/track2_final_dictionary_neural_experiments')
EXPERIMENT_ROOT.mkdir(parents=True,exist_ok=True)
NEURAL_CHECKPOINT=Path('/content/drive/MyDrive/PHD/Datathon/track2_phase2_v36_phase1_teacher_demucs/run_v1/teacher_epoch_01.pt')
NEURAL_ALPHA=.95
NEURAL_TAG='v36_noise_adaptive_residual_demucs_dns64'
NEURAL_SEGMENT_SECONDS=3.2;NEURAL_HOP_SECONDS=1.6
NEURAL_SIGN_ENSEMBLE=True
if not NEURAL_CHECKPOINT.is_file():raise FileNotFoundError(NEURAL_CHECKPOINT)
print('Neural fallback:',NEURAL_CHECKPOINT,'alpha',NEURAL_ALPHA)
print('Existing model libraries are reused. No forced version changes.')


Mounted at /content/drive
V24 Phase-2 full released-bank refit
Phase-2 input: /content/drive/MyDrive/PHD/Datathon/input_data.zip
Expected clips: 3178
V5 artifacts: /content/drive/MyDrive/Datathon/work/v5_synthetic_residual_evidence
Frozen V6 run: /content/drive/MyDrive/Datathon/track2_v6_direct_reconstruction/run_23523d3d7e5ed3ea
Neural fallback: /content/drive/MyDrive/PHD/Datathon/track2_phase2_v36_phase1_teacher_demucs/run_v1/teacher_epoch_01.pt alpha 0.95
Existing model libraries are reused. No forced version changes.


## 2. Exact uploaded dictionary matcher and reconstruction code

In [ ]:
"""Full-clip residual-template detector. No query labels/clean audio in inference."""
from __future__ import annotations
import os, io, re, json, math, time, hashlib, pickle, zipfile, copy
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import OrderedDict, Counter
import numpy as np
import torch
import torch.nn.functional as F
import soundfile as sf

VERSION = 'residual_evidence_v5_20260917_r1'

@dataclass(frozen=True)
class MatcherConfig:
    sample_rate: int = 16000
    coarse_rate: int = 4000
    coarse_floor: float = 0.25
    template_batch: int = 32
    peaks_per_template: int = 3
    verify_topk: int = 128
    raw_radius: int = 12
    min_template_sec: float = 0.08
    max_template_sec: float = 6.0
    max_templates: int = 2000
    chunk_sec: float = 24.0
    fft_cache_mb: int = 384
    merge_iou: float = 0.65
    seed: int = 517


def digest(x):
    return hashlib.sha256(json.dumps(x, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()).hexdigest()


def file_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(4 << 20), b''):
            h.update(b)
    return h.hexdigest()


def wave_sha(y):
    return hashlib.sha256(np.asarray(y, '<f4').tobytes()).hexdigest()


def atomic_json(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + '.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, allow_nan=False)
    os.replace(tmp, p)


def atomic_pickle(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + '.tmp')
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, p)


def event_list(v):
    if isinstance(v, dict):
        if 'events' not in v:
            raise ValueError('Prediction row has no events field.')
        v = v['events']
    out = []
    for e in v:
        if isinstance(e, dict):
            a = float(e['onset'] if 'onset' in e else e['start'])
            b = float(e['offset'] if 'offset' in e else e['end'])
        else:
            a, b = float(e[0]), float(e[1])
        if not (math.isfinite(a) and math.isfinite(b) and 0 <= a < b):
            raise ValueError(f'Invalid event: {e}')
        out.append((a, b))
    return out


def read_jsonl(path, predictions=True):
    out = {}
    with open(path, encoding='utf-8') as f:
        for n, line in enumerate(f, 1):
            if not line.strip(): continue
            r = json.loads(line)
            cid = str(r['clip_id'])
            if cid in out: raise ValueError(f'Duplicate clip_id on line {n}: {cid}')
            if predictions: event_list(r)
            out[cid] = r
    if not out: raise ValueError(f'Empty JSONL: {path}')
    return out


def source_group(cid):
    # Best-effort recording prefix, NOT a claim of verified speaker identity.
    # Example: ...844425030213767_010_LowerDibangvalley-SPECIFIC_...
    m = re.search(r'_\d+_(?:GENERIC|[^_]+-SPECIFIC)_', str(cid))
    return str(cid)[:m.start()] if m else str(cid).rsplit('_', 3)[0]


def split_bucket(group):
    return int(hashlib.sha256(('v5_bank_source_' + group).encode()).hexdigest()[:12], 16) % 10


def bandlimited_resample(y, src, dst):
    """Windowed-sinc resampling, no SciPy/librosa dependency; common grid for pairs."""
    y = np.asarray(y, np.float32)
    if src == dst: return y.copy()
    n = int(round(len(y) * dst / src))
    if n < 1: raise ValueError('Audio became empty during resampling.')
    cutoff = min(1.0, float(dst) / src) * 0.94
    radius = int(math.ceil(20 / cutoff))
    offsets = np.arange(-radius, radius + 1)
    out = np.empty(n, np.float32)
    for a in range(0, n, 2048):
        b = min(n, a + 2048)
        pos = np.arange(a, b, dtype=np.float64) * src / dst
        ix = np.floor(pos).astype(np.int64)[:, None] + offsets[None]
        distance = pos[:, None] - ix
        w = cutoff * np.sinc(cutoff * distance)
        w *= np.where(np.abs(distance) < radius, 0.5 + 0.5*np.cos(np.pi*distance/radius), 0)
        valid = (ix >= 0) & (ix < len(y))
        w *= valid
        w /= np.maximum(w.sum(1, keepdims=True), 1e-12)
        out[a:b] = np.sum(y[np.clip(ix, 0, len(y)-1)] * w, axis=1)
    return out


def load_audio(path_or_bytes, sr=16000):
    obj = io.BytesIO(path_or_bytes) if isinstance(path_or_bytes, (bytes, bytearray)) else path_or_bytes
    y, fs = sf.read(obj, dtype='float32', always_2d=True)
    y = np.mean(y, axis=1, dtype=np.float32)
    if not len(y) or not np.isfinite(y).all(): raise ValueError(f'Invalid audio: {str(path_or_bytes)[:100]}')
    return bandlimited_resample(y, int(fs), sr)


def condition_wave(y):
    """Mild pre-emphasis, identically applied to templates and query recordings."""
    y = np.asarray(y, np.float32)
    return np.concatenate([y[:1], y[1:] - 0.85*y[:-1]]).astype(np.float32)


def coarse_wave(y, cfg):
    ratio = cfg.sample_rate // cfg.coarse_rate
    if ratio * cfg.coarse_rate != cfg.sample_rate:
        raise ValueError('coarse_rate must divide sample_rate.')
    y = condition_wave(y)
    t = np.arange(-32, 33, dtype=np.float64)
    cutoff = 0.90 / ratio
    kernel = cutoff * np.sinc(cutoff*t) * np.hanning(len(t))
    kernel /= kernel.sum()
    # Odd, symmetric FIR; zero padding preserves the time origin.
    padded = np.pad(y, (32, 32))
    return np.convolve(padded, kernel, mode='valid')[::ratio].astype(np.float32)


def iou(a, b):
    inter = max(0., min(a[1], b[1]) - max(a[0], b[0]))
    union = max(a[1], b[1]) - min(a[0], b[0])
    return inter / union if union > 0 else 0.


def closest_first_pairs(pred, ref):
    # Same explicitly named local scorer used in the user's notebooks.
    edges = []
    for i, (a,b) in enumerate(pred):
        for j, (s,e) in enumerate(ref):
            tol = max(0.20 * (e-s), 0.05)
            if abs(a-s) <= tol and abs(b-e) <= tol:
                edges.append((abs(a-s) + abs(b-e), i, j))
    used_p, used_r, pairs = set(), set(), []
    for _, i, j in sorted(edges):
        if i not in used_p and j not in used_r:
            used_p.add(i); used_r.add(j); pairs.append((i,j))
    return pairs


def score_events(pred, refs, durs, ids):
    tp = fp = fn = 0; dice = []; npred = ngt = 0
    for cid in ids:
        p, g = event_list(pred[cid]), event_list(refs[cid])
        nmatch = len(closest_first_pairs(p, g))
        tp += nmatch; fp += len(p)-nmatch; fn += len(g)-nmatch
        npred += len(p); ngt += len(g)
        n = max(1, int(round(durs[cid]/0.01)))
        pm = np.zeros(n, bool); gm = np.zeros(n, bool)
        for events, mask in ((p,pm),(g,gm)):
            for a,b in events:
                lo,hi = max(0,int(round(a/0.01))), min(n,int(round(b/0.01)))
                if hi > lo: mask[lo:hi] = True
        den = int(pm.sum()+gm.sum())
        dice.append(1. if den == 0 else 2.*np.logical_and(pm,gm).sum()/den)
    precision = tp/(tp+fp) if tp+fp else 0.
    recall = tp/(tp+fn) if tp+fn else 0.
    f1 = 2.*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
    d = float(np.mean(dice)) if dice else 0.
    return dict(f1=f1, precision=precision, recall=recall, dice=d, combined=f1+d,
                tp=tp, fp=fp, fn=fn, clips=len(ids), predicted_events=npred, gt_events=ngt)


def wilson_lower(success, total, z=1.96):
    if total == 0: return 0.
    p = success/total; den = 1.+z*z/total
    return (p+z*z/(2*total)-z*math.sqrt(p*(1-p)/total+z*z/(4*total*total)))/den


def build_bank(bank_ids, refs, audio_paths, clean_paths, groups, cfg, progress=lambda x, **k: x):
    records, audits, source_hashes = [], [], []
    sr = cfg.sample_rate
    for cid in progress(bank_ids, desc='BANK pairs only: noisy minus clean'):
        x = load_audio(audio_paths[cid], sr)
        s = load_audio(clean_paths[cid], sr)
        if len(x) != len(s):
            raise ValueError(f'Pair length mismatch: {cid}; no silent truncation allowed.')
        mask = np.zeros(len(x), bool)
        for a,b in refs[cid]:
            mask[max(0,int(round(a*sr))):min(len(x),int(round(b*sr)))] = True
        outside = ~mask
        # Gain alignment is estimated from BANK-only non-event samples.
        scale = float(np.dot(x[outside],s[outside])/(np.dot(s[outside],s[outside])+1e-12)) if outside.sum()>sr//10 else 1.
        scale = float(np.clip(scale,0.8,1.2))
        residual = x - scale*s
        inside_rms = float(np.sqrt(np.mean(residual[mask]**2)+1e-20)) if mask.any() else 0.
        outside_rms = float(np.sqrt(np.mean(residual[outside]**2)+1e-20)) if outside.any() else 0.
        ratio = outside_rms/max(inside_rms,1e-10)
        good = bool(inside_rms>1e-5 and ratio<=0.15)
        audits.append(dict(cid=cid, scale=scale, inside_rms=inside_rms, outside_rms=outside_rms,
                           outside_inside_ratio=ratio, used=good))
        source_hashes.append(wave_sha(x))
        if not good: continue
        for j,(a,b) in enumerate(refs[cid]):
            # Mixed overlapping events are not single-template examples.
            if any(k!=j and min(b,e)-max(a,s0)>0.005 for k,(s0,e) in enumerate(refs[cid])): continue
            if not cfg.min_template_sec <= b-a <= cfg.max_template_sec: continue
            lo,hi = int(round(a*sr)), int(round(b*sr))
            if lo < 0 or hi > len(x) + int(.02*sr): raise ValueError(f'BANK label out of audio range: {cid}')
            hi = min(hi, len(x))
            w = residual[lo:hi].astype(np.float32)
            if len(w)<16 or float(np.std(w))<1e-5: continue
            coarse = coarse_wave(w,cfg)
            coarse -= coarse.mean()
            norm = float(np.linalg.norm(coarse))
            if norm < 1e-7: continue
            raw = condition_wave(w)
            # Reject templates with effectively just one energetic quarter.
            chunks = np.array([np.sum(v*v) for v in np.array_split(raw,4)])
            if np.sum(chunks > .02*max(float(chunks.sum()),1e-10)) < 2: continue
            records.append(dict(tid=f'{cid}::{j}', source_cid=cid, group=groups[cid],
                                wave=w, raw=raw, coarse=(coarse/norm).astype(np.float32),
                                annotation_duration=b-a))
    # Exact waveform deduplication only; no fitting on SELECT/HOLD.
    unique = {}; dedup = 0
    for r in records:
        w = r['wave']/max(float(np.max(np.abs(r['wave']))),1e-9)
        key = hashlib.sha256(np.round(w*32700).astype('<i2').tobytes()).hexdigest()
        if key in unique: dedup += 1
        else: unique[key] = r
    records = sorted(unique.values(), key=lambda r: hashlib.sha256(r['tid'].encode()).hexdigest())
    capped = max(0,len(records)-cfg.max_templates)
    records = records[:cfg.max_templates]
    if not records: raise RuntimeError('No valid additive-noise templates. Stop: the waveform-template hypothesis is unsupported for these pairs.')
    return dict(version=VERSION, cfg=asdict(cfg), records=records, audits=audits,
                bank_ids=list(bank_ids), bank_groups=sorted({groups[c] for c in bank_ids}),
                source_hashes=source_hashes, deduplicated=dedup, capped=capped)


class ResidualMatcher:
    def __init__(self, bank, cfg, device=None):
        self.cfg, self.bank = cfg, bank
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.records = bank['records']
        self.source_hashes = set(bank.get('source_hashes',[]))
        self.cache = OrderedDict(); self.cache_bytes = 0
        self.signature = digest(dict(version=VERSION,cfg=asdict(cfg),
            templates=[(r['tid'], wave_sha(r['wave'])) for r in self.records]))

    def _spectrum(self, ix, nfft):
        key = (tuple(ix), nfft)
        if key in self.cache:
            z = self.cache.pop(key); self.cache[key] = z
            return z
        t = torch.zeros((len(ix),nfft),dtype=torch.float32,device=self.device)
        for j,i in enumerate(ix):
            x = self.records[i]['coarse']; t[j,:len(x)] = torch.as_tensor(x,device=self.device)
        z = torch.fft.rfft(t,n=nfft,dim=-1)
        cost = z.numel()*z.element_size(); max_bytes = self.cfg.fft_cache_mb*(1<<20)
        while self.cache and self.cache_bytes+cost > max_bytes:
            _,old = self.cache.popitem(last=False); self.cache_bytes-=old.numel()*old.element_size()
        if cost<=max_bytes: self.cache[key] = z; self.cache_bytes+=cost
        return z

    @torch.inference_mode()
    def _coarse_scan(self,y,exclude_group=None):
        x = coarse_wave(y,self.cfg); n = len(x)
        eligible = [i for i,r in enumerate(self.records)
                    if len(r['coarse'])<=n and (exclude_group is None or r['group']!=exclude_group)]
        if not eligible: return []
        mmax = max(len(self.records[i]['coarse']) for i in eligible)
        nfft = 1<<(n+mmax-2).bit_length()
        xt = torch.as_tensor(x,dtype=torch.float32,device=self.device)
        xf = torch.fft.rfft(xt,n=nfft)
        # Use float64 prefix sums to avoid variance cancellation near silence.
        xx = xt.double()
        pref = torch.cat([torch.zeros(1,dtype=torch.float64,device=self.device),xx.cumsum(0)])
        pref2 = torch.cat([torch.zeros(1,dtype=torch.float64,device=self.device),(xx*xx).cumsum(0)])
        starts = torch.arange(n,device=self.device)[None]
        rows = []
        for a in range(0,len(eligible),self.cfg.template_batch):
            ix = eligible[a:a+self.cfg.template_batch]
            lens = torch.tensor([len(self.records[i]['coarse']) for i in ix],device=self.device)[:,None]
            tf = self._spectrum(ix,nfft)
            dot = torch.fft.irfft(xf[None]*tf.conj(),n=nfft,dim=-1)[:,:n]
            ends = (starts+lens).clamp(max=n)
            total = pref[ends]-pref[starts]; total2 = pref2[ends]-pref2[starts]
            energy = (total2-total*total/lens).clamp(min=1e-16).float()
            q = (dot.abs()/torch.sqrt(energy)).clamp(max=1.)
            q = q.masked_fill((starts+lens>n)|(energy<1e-12),-1.)
            for _ in range(self.cfg.peaks_per_template):
                vals, pos = q.max(dim=1)
                for j,(v,p) in enumerate(zip(vals.cpu().tolist(),pos.cpu().tolist())):
                    if v>=self.cfg.coarse_floor:
                        rows.append(dict(template_index=ix[j], sample4k=int(p), coarse=float(v)))
                rad = torch.maximum(lens//2,torch.full_like(lens,int(.06*self.cfg.coarse_rate)))
                q = q.masked_fill((starts-pos[:,None]).abs()<=rad,-1.)
        return sorted(rows,key=lambda r:r['coarse'],reverse=True)[:self.cfg.verify_topk]

    def _verify(self,raw_query,proposal):
        r = self.records[proposal['template_index']]; t = r['raw']; m = len(t)
        center = proposal['sample4k']*(self.cfg.sample_rate//self.cfg.coarse_rate)
        lo,hi = max(0,center-self.cfg.raw_radius), min(len(raw_query)-m,center+self.cfg.raw_radius)
        if hi<lo: return None
        t0=t.astype(np.float64)-np.mean(t,dtype=np.float64); ten=float(t0@t0)
        if ten<1e-12: return None
        seg=raw_query[lo:hi+m].astype(np.float64)
        dots=np.correlate(seg,t0,mode='valid')
        cs=np.r_[0.,np.cumsum(seg)]; cs2=np.r_[0.,np.cumsum(seg*seg)]
        sums=cs[m:]-cs[:-m]; energies=np.maximum(cs2[m:]-cs2[:-m]-sums*sums/m,1e-15)
        corr=dots/np.sqrt(energies*ten)
        j=int(np.argmax(np.abs(corr))); start=lo+j; sign=1. if corr[j]>=0 else -1.
        qseg=raw_query[start:start+m]
        chunks=[]
        for inds in np.array_split(np.arange(m),4):
            tt=t[inds].astype(np.float64); yy=qseg[inds].astype(np.float64)
            tt-=tt.mean(); yy-=yy.mean(); et=float(tt@tt)
            if et < .015*ten: continue
            chunks.append(float(sign*np.dot(tt,yy)/math.sqrt(max(et*float(yy@yy),1e-20))))
        if len(chunks)<2: return None
        raw_corr=float(abs(corr[j])); chunk=float(np.median(chunks))
        quality=float(np.clip(min(raw_corr,chunk),0.,1.))
        return dict(onset=start/self.cfg.sample_rate,offset=(start+m)/self.cfg.sample_rate,
                    quality=quality,corr16=raw_corr,corr4k=proposal['coarse'],chunk_median=chunk,
                    template_id=r['tid'],template_source=r['source_cid'],polarity=int(sign))

    def scan(self,y,exclude_group=None):
        """Only noisy waveform and a training bank are inputs. No event/GT arguments."""
        y=np.asarray(y,np.float32)
        if not np.isfinite(y).all(): raise ValueError('Nonfinite query audio.')
        if wave_sha(y) in self.source_hashes:
            raise RuntimeError('Query waveform exactly duplicates a BANK source. Refusing self-retrieval.')
        chunk=int(self.cfg.chunk_sec*self.cfg.sample_rate)
        overlap=int((self.cfg.max_template_sec+.05)*self.cfg.sample_rate)
        if chunk<=overlap: raise ValueError('chunk_sec must exceed max_template_sec.')
        starts=list(range(0,max(1,len(y)-overlap),chunk-overlap))
        found=[]
        for a in starts:
            yy=y[a:min(len(y),a+chunk)]
            if len(yy)<int(self.cfg.min_template_sec*self.cfg.sample_rate): continue
            raw=condition_wave(yy)
            for p in self._coarse_scan(yy,exclude_group):
                v=self._verify(raw,p)
                if v is not None:
                    v['onset']+=a/self.cfg.sample_rate; v['offset']+=a/self.cfg.sample_rate
                    found.append(v)
        # Keep geometric alternatives; final threshold/NMS is applied later.
        best={}
        for r in found:
            k=(round(r['onset'],4),round(r['offset'],4),r['template_id'])
            if k not in best or r['quality']>best[k]['quality']: best[k]=r
        return sorted(best.values(),key=lambda r:r['quality'],reverse=True)


def select_matches(rows, threshold, duplicate_iou=.65):
    chosen=[]
    for r in sorted(rows,key=lambda z:z['quality'],reverse=True):
        if r['quality']<threshold: continue
        ev=(r['onset'],r['offset'])
        if any(iou(ev,(x['onset'],x['offset']))>=duplicate_iou or
               (abs(ev[0]-x['onset'])<=.02 and abs(ev[1]-x['offset'])<=.02) for x in chosen): continue
        chosen.append(r)
    return sorted(chosen,key=lambda z:(z['onset'],z['offset']))


def patch_events(base, matches, max_move=.25, minimum_iou=.50, allow_add=False, add_threshold=.85):
    """Conservative edit: only mutual one-to-one overlaps; unmatched base retained."""
    base=event_list(base); proposal=[(m['onset'],m['offset']) for m in matches]
    edges=[]
    p2b={j:[] for j in range(len(proposal))}; b2p={i:[] for i in range(len(base))}
    for j,p in enumerate(proposal):
        for i,b in enumerate(base):
            ov=iou(p,b)
            if ov>=minimum_iou and max(abs(p[0]-b[0]),abs(p[1]-b[1]))<=max_move:
                p2b[j].append(i); b2p[i].append(j); edges.append((i,j))
    out=list(base); used=set(); changed=0
    for i,j in edges:
        if len(p2b[j])==len(b2p[i])==1:
            out[i]=proposal[j]; used.add(j); changed+=int(out[i]!=base[i])
    added=0
    if allow_add:
        for j,p in enumerate(proposal):
            if j in used or matches[j]['quality']<add_threshold: continue
            # No addition overlapping or touching an existing interval.
            if any(min(p[1],b[1]) - max(p[0],b[0]) > -.04 for b in out): continue
            out.append(p); added+=1
    return sorted(out),dict(changed=changed,added=added)


class WaveSource:
    """Read WAVs from one explicit folder/ZIP without extracting or guessing IDs."""
    def __init__(self,path,required_ids):
        self.path=str(path); self.archive=None; self.index={}
        if Path(path).is_file() and zipfile.is_zipfile(path):
            self.archive=zipfile.ZipFile(path)
            names=[i.filename for i in self.archive.infolist() if not i.is_dir() and i.filename.lower().endswith('.wav') and '__MACOSX' not in i.filename]
        elif Path(path).is_dir():
            names=[str(p) for p in Path(path).rglob('*.wav')]
        else: raise FileNotFoundError(path)
        for name in names:
            cid=Path(name).stem
            if cid not in required_ids: continue
            if cid in self.index: raise ValueError(f'Duplicate waveform stem in {path}: {cid}')
            self.index[cid]=name
        missing=set(required_ids)-set(self.index)
        if missing: raise ValueError(f'{path}: missing {len(missing)} required WAVs; examples {sorted(missing)[:3]}')
    def load(self,cid,sr=16000):
        loc=self.index[cid]
        return load_audio(self.archive.read(loc) if self.archive is not None else loc,sr)
    def duration(self,cid):
        loc=self.index[cid]
        x=io.BytesIO(self.archive.read(loc)) if self.archive is not None else loc
        info=sf.info(x)
        return info.frames/info.samplerate


def persist_query(matcher,y,path,exclude_group=None):
    wh=wave_sha(y); path=Path(path)
    signature=digest(dict(matcher=matcher.signature,wave=wh,exclude_group=exclude_group))
    if path.is_file():
        with open(path) as f: old=json.load(f)
        if old.get('signature')==signature: return old['proposals']
    rows=matcher.scan(y,exclude_group)
    atomic_json(path,dict(signature=signature,query_wave_sha256=wh,proposals=rows))
    return rows


def unit_tests():
    cfg=MatcherConfig(coarse_floor=.15,template_batch=2,peaks_per_template=2,verify_topk=20,fft_cache_mb=16)
    rng=np.random.default_rng(601)
    sr=cfg.sample_rate
    t=(rng.normal(0,.15,int(.36*sr))*np.hanning(int(.36*sr))).astype(np.float32)
    source=rng.normal(0,.02,int(1.8*sr)).astype(np.float32)
    a=1923; source[a:a+len(t)]+=t
    coarse=coarse_wave(t,cfg); coarse-=coarse.mean(); coarse/=np.linalg.norm(coarse)
    bank=dict(records=[dict(tid='test_template',source_cid='bank',group='g0',wave=t,raw=condition_wave(t),coarse=coarse)],source_hashes=[])
    m=ResidualMatcher(bank,cfg,device='cpu')
    got=select_matches(m.scan(source),.85)
    assert got and abs(got[0]['onset']-a/sr)<=1/sr, got
    assert abs(got[0]['offset']-(a+len(t))/sr)<=1/sr
    neg=m.scan(rng.normal(0,.02,len(source)).astype(np.float32))
    assert not select_matches(neg,.85)
    # Confirm no self-retrieval.
    m.source_hashes.add(wave_sha(source))
    try: m.scan(source)
    except RuntimeError: pass
    else: raise AssertionError('Self-retrieval guard failed.')
    refs={'x':[(1.,1.5)]}; durs={'x':2.}
    assert score_events(refs,refs,durs,['x'])['f1']==1.
    assert score_events({'x':[]},refs,durs,['x'])['fn']==1
    row=dict(onset=1.01,offset=1.51,quality=.99)
    patched,diag=patch_events([(1.,1.5)],[row])
    assert len(patched)==1 and diag['changed']==1
    patched,diag=patch_events([(1.,1.5)],[dict(onset=0.,offset=.3,quality=.99)])
    assert patched==[(1.,1.5)] and diag['added']==0
    print('PASS: FFT timing, raw sample alignment, negative audio, self-retrieval guard, scorer, conservative edits.')
    return True

In [ ]:
"""Track-2 direct residual reconstruction. Inference never accepts a clean reference."""
from __future__ import annotations
import copy, hashlib, io, json, math, os, re, shutil, time, unicodedata, wave, zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np
import soundfile as sf

RECON_VERSION = 'track2_v6_direct_residual_20260918_r1'

@dataclass(frozen=True)
class ReconConfig:
    sample_rate: int = 16000
    max_components: int = 16
    duplicate_iou: float = 0.65
    alignment_radius: int = 4
    min_abs_gain: float = 0.01
    max_abs_gain: float = 8.0
    max_condition: float = 1e7
    max_gain_disagreement: float = 0.50
    max_removal_energy_ratio: float = 1.15
    max_output_energy_ratio: float = 1.05
    max_natural_modified_fraction: float = 0.05
    ridge_relative: float = 1e-9
    whitening_floor_fraction: float = 0.02
    gls_iterations: int = 3
    natural_damage_db: float = -24.0


def jsonable(x):
    if isinstance(x, dict): return {str(k):jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [jsonable(v) for v in x]
    if isinstance(x, Path): return str(x)
    if isinstance(x, np.ndarray): return jsonable(x.tolist())
    if isinstance(x, np.generic): return jsonable(x.item())
    if isinstance(x, float) and not math.isfinite(x): return None
    return x


def put_json(path, obj):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    t=p.with_name(p.name+'.tmp')
    with open(t,'w',encoding='utf-8') as f:
        json.dump(jsonable(obj),f,indent=2,ensure_ascii=False,allow_nan=False)
    os.replace(t,p)


def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4<<20),b''): h.update(b)
    return h.hexdigest()


def sha_json(obj):
    return hashlib.sha256(json.dumps(jsonable(obj),sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()


def canonical_pcm(y):
    a=np.asarray(y,np.float64)
    if a.ndim!=1 or not len(a) or not np.isfinite(a).all():
        raise ValueError('Wave must be finite, nonempty, and mono.')
    # Deterministic rounding, not peak normalization. Used for metrics AND export.
    return np.clip(np.rint(a*32768.0),-32768,32767).astype('<i2')


def pcm_roundtrip(y): return canonical_pcm(y).astype(np.float64)/32768.0


def wav_bytes(y, sr=16000):
    buf=io.BytesIO()
    with wave.open(buf,'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes(canonical_pcm(y).tobytes())
    return buf.getvalue()


def write_pcm(path,y,sr=16000):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    data=wav_bytes(y,sr); tmp=p.with_name(p.name+'.tmp')
    with open(tmp,'wb') as f: f.write(data)
    os.replace(tmp,p)
    return hashlib.sha256(data).hexdigest()


def read_wave_strict(path_or_bytes,sr=16000):
    source=io.BytesIO(path_or_bytes) if isinstance(path_or_bytes,(bytes,bytearray)) else path_or_bytes
    y,fs=sf.read(source,dtype='float32',always_2d=True)
    if fs!=sr or y.shape[1]!=1:
        raise ValueError(f'Expected native {sr}-Hz mono audio, got {fs} Hz / {y.shape[1]} channels. No unvalidated resampling.')
    a=y[:,0]
    if not len(a) or not np.isfinite(a).all(): raise ValueError('Empty or invalid audio.')
    return a


def wav_audit(path_or_bytes,expected_frames,sr=16000):
    data=path_or_bytes if isinstance(path_or_bytes,bytes) else Path(path_or_bytes).read_bytes()
    with wave.open(io.BytesIO(data),'rb') as w:
        got=(w.getframerate(),w.getnchannels(),w.getsampwidth(),w.getnframes(),w.getcomptype())
        wanted=(sr,1,2,int(expected_frames),'NONE')
        if got!=wanted: raise ValueError(f'WAV compliance error: {got} != {wanted}')
        pcm=w.readframes(expected_frames)
        if len(pcm)!=expected_frames*2: raise ValueError('Truncated PCM payload.')
    return hashlib.sha256(data).hexdigest()


class StrictWaveSource:
    """One explicit directory/ZIP, exact IDs; no duplicate stems or inferred paths."""
    def __init__(self,path,required_ids):
        self.path=str(path); self.archive=None; self.index={}
        required_ids=set(map(str,required_ids))
        p=Path(path)
        if p.is_file() and zipfile.is_zipfile(p):
            self.archive=zipfile.ZipFile(p)
            names=[z.filename for z in self.archive.infolist() if not z.is_dir() and z.filename.lower().endswith('.wav') and '__MACOSX' not in z.filename]
        elif p.is_dir(): names=[str(q) for q in p.rglob('*') if q.is_file() and q.suffix.lower()=='.wav']
        else: raise FileNotFoundError(p)
        for name in names:
            cid=Path(name).stem
            if cid not in required_ids: continue
            if cid in self.index: raise ValueError(f'Duplicate waveform ID: {cid} in {p}')
            self.index[cid]=name
        missing=required_ids-set(self.index)
        if missing: raise ValueError(f'{p}: {len(missing)} missing IDs; examples {sorted(missing)[:3]}')
    def bytes(self,cid):
        loc=self.index[cid]
        return self.archive.read(loc) if self.archive is not None else Path(loc).read_bytes()
    def load(self,cid): return read_wave_strict(self.bytes(cid))
    def close(self):
        if self.archive is not None: self.archive.close()


def si_sdr(estimate,reference):
    """Zero-mean SI-SDR, no alignment/resampling; 120-dB numerical ceiling only."""
    x=np.asarray(estimate,np.float64); s=np.asarray(reference,np.float64)
    if x.shape!=s.shape or x.ndim!=1: raise ValueError('SI-SDR length/shape mismatch.')
    if not np.isfinite(x).all() or not np.isfinite(s).all(): raise ValueError('Nonfinite SI-SDR input.')
    x=x-x.mean(); s=s-s.mean(); ss=float(s@s)
    if ss<=1e-18: raise ValueError('Silent reference: SI-SDR undefined, not silently excluded.')
    target=(float(x@s)/ss)*s; e=x-target
    num=float(target@target); den=float(e@e)
    value=10*math.log10(max(num,ss*1e-16)/max(den,ss*1e-12))
    return float(min(120.,value))


def residual_change_db(a,b):
    a=np.asarray(a,np.float64); b=np.asarray(b,np.float64)
    return float(10*np.log10(max(float((a-b)@(a-b)),1e-24)/max(float(b@b),1e-24)))


def fit_centered(A,y,weights=None,ridge=1e-9):
    A=np.asarray(A,np.float64); y=np.asarray(y,np.float64)
    if A.ndim==1: A=A[:,None]
    w=np.ones(len(y),np.float64) if weights is None else np.asarray(weights,np.float64)
    w=np.clip(w,0,None); w=w/max(float(w.mean()),1e-30)
    am=(A*w[:,None]).sum(0)/max(w.sum(),1e-30); ym=float((y*w).sum()/max(w.sum(),1e-30))
    B=A-am; v=y-ym
    gram=B.T@(w[:,None]*B); rhs=B.T@(w*v)
    diag=np.diag(gram); scale=np.sqrt(np.maximum(diag,1e-30))
    G=gram/scale[:,None]/scale[None,:]; r=rhs/scale
    cond=float(np.linalg.cond(G))
    g=np.linalg.solve(G+ridge*np.eye(G.shape[0]),r)/scale
    return g,cond


def emphasis(A,y):
    return A[1:]-.90*A[:-1],y[1:]-.90*y[:-1]


def residual_psd(residual,nfft,sr=16000):
    nwin=min(512,len(residual))
    if nwin<16: return np.ones(nfft//2+1)
    hop=max(1,nwin//2); win=np.hanning(nwin)
    starts=range(0,len(residual)-nwin+1,hop)
    # Median-Welch PSD limits unusually loud speech frames without target labels.
    frames=np.stack([residual[a:a+nwin]*win for a in starts])
    p=np.median(np.abs(np.fft.rfft(frames,n=512,axis=1))**2,axis=0)
    p=np.convolve(np.pad(p,(3,3),mode='edge'),np.ones(7)/7,mode='valid')
    return np.interp(np.linspace(0,sr/2,nfft//2+1),np.linspace(0,sr/2,len(p)),p)


def fit_spectral(A,y,cfg):
    g,_=fit_centered(*emphasis(A,y),ridge=cfg.ridge_relative)
    nfft=1<<(len(y)-1).bit_length()
    AF=np.fft.rfft(A,n=nfft,axis=0); YF=np.fft.rfft(y,n=nfft)
    freq=np.linspace(0,cfg.sample_rate/2,len(YF))
    cond=1.
    for _ in range(cfg.gls_iterations):
        psd=residual_psd(y-A@g,nfft,cfg.sample_rate)
        positive=psd[psd>0]
        floor=max(float(np.median(positive))*cfg.whitening_floor_fraction if len(positive) else 0.,1e-20)
        w=1./np.maximum(psd,floor); w/=max(float(np.median(w)),1e-30)
        w=np.clip(w,1e-3,1e3); w[freq<70.]=0.
        # One-sided real FFT Parseval weights.
        w[1:-1]*=2.
        gram=(AF.conj().T@(w[:,None]*AF)).real
        rhs=(AF.conj().T@(w*YF)).real
        scale=np.sqrt(np.maximum(np.diag(gram),1e-30))
        G=gram/scale[:,None]/scale[None,:]
        cond=float(np.linalg.cond(G))
        g=np.linalg.solve(G+cfg.ridge_relative*np.eye(len(g)),rhs/scale)/scale
    return g,cond


def estimate_gains(A,y,polarity,method,cfg):
    if method=='stored_gain': return np.asarray(polarity,np.float64),1.
    if method=='mixture_ols': return fit_centered(A,y,ridge=cfg.ridge_relative)
    P,v=emphasis(A,y)
    if method=='preemphasis_ols': return fit_centered(P,v,ridge=cfg.ridge_relative)
    if method=='spectral_gls': return fit_spectral(A,y,cfg)
    if method=='block_irls':
        g,cond=fit_centered(P,v,ridge=cfg.ridge_relative)
        block=max(64,int(.030*cfg.sample_rate))
        idx=np.arange(len(v))//block
        for _ in range(cfg.gls_iterations):
            residual=v-P@g
            counts=np.bincount(idx); power=np.bincount(idx,weights=residual**2)/counts
            floor=max(float(np.median(power))*.05,1e-15)
            bw=1./np.maximum(power,floor); bw/=max(float(np.median(bw)),1e-30)
            weights=np.clip(bw,0.02,50.)[idx]
            g,cond=fit_centered(P,v,weights,cfg.ridge_relative)
        return g,cond
    raise ValueError(f'Unknown deployable gain method: {method}')


def gain_disagreement(A,y,g,cfg):
    # Two interleaved block fits: agreement diagnostic, not a probability of correctness.
    P,v=emphasis(A,y); block=max(64,int(.020*cfg.sample_rate))
    parity=(np.arange(len(v))//block)%2
    folds=[]
    for f in (0,1):
        mask=parity==f
        if mask.sum()<4*A.shape[1] or np.any((P[mask]**2).sum(0)<1e-14): return math.inf
        try: gg,cc=fit_centered(P[mask],v[mask],ridge=cfg.ridge_relative)
        except np.linalg.LinAlgError: return math.inf
        if cc>cfg.max_condition: return math.inf
        folds.append(gg)
    return float(np.max(np.abs(folds[0]-folds[1])/np.maximum(np.abs(g),.025)))


def refine_placement(y,t,start,radius=4):
    """Integer-sample correlation refinement. The raw template is never faded/stretched."""
    t=np.asarray(t,np.float64); dt=t[1:]-.9*t[:-1]; dt=dt-dt.mean()
    ten=float(dt@dt)
    if ten<1e-18: return start
    best=(-math.inf,start)
    for a in range(max(0,start-radius),min(len(y)-len(t),start+radius)+1):
        seg=np.asarray(y[a:a+len(t)],np.float64); ds=seg[1:]-.9*seg[:-1]; ds-=ds.mean()
        q=abs(float(ds@dt))/math.sqrt(max(ten*float(ds@ds),1e-30))
        # Stable tie-breaking favours the verified V5 start.
        score=q-1e-12*abs(a-start)
        if score>best[0]: best=(score,a)
    return best[1]


def interval_iou(a,b):
    inter=max(0,min(a[1],b[1])-max(a[0],b[0])); union=max(a[1],b[1])-min(a[0],b[0])
    return inter/union if union>0 else 0.


def prepared_matches(y,proposals,bank_by_id,cfg):
    """Prepare ALL verified alternatives once; threshold/NMS are applied afterwards."""
    out=[]
    for p in proposals:
        tid=p['template_id']
        if tid not in bank_by_id: raise ValueError(f'Unknown cached template {tid}; wrong bank.')
        t=np.asarray(bank_by_id[tid]['wave'],np.float64)
        start=int(round(float(p['onset'])*cfg.sample_rate))
        if start<0 or start+len(t)>len(y): continue
        if abs((float(p['offset'])-float(p['onset']))*cfg.sample_rate-len(t))>2:
            raise ValueError('Match duration is incompatible with frozen template length.')
        a=refine_placement(y,t,start,cfg.alignment_radius)
        out.append(dict(start=int(a),end=int(a+len(t)),quality=float(p['quality']),
                        polarity=int(p['polarity']),template_id=tid,wave=t))
    return out


def select_evidence(prepared,threshold,cfg):
    keep=[]
    for r in sorted(prepared,key=lambda q:q['quality'],reverse=True):
        if r['quality']<threshold: continue
        if any(interval_iou((r['start'],r['end']),(v['start'],v['end']))>=cfg.duplicate_iou for v in keep): continue
        keep.append(r)
        if len(keep)>=cfg.max_components: break
    return sorted(keep,key=lambda q:q['start'])


def overlap_components(rows):
    groups=[]
    for r in sorted(rows,key=lambda q:q['start']):
        if groups and r['start']<max(x['end'] for x in groups[-1]): groups[-1].append(r)
        else: groups.append([r])
    return groups


def component_matrix(y,rows):
    lo=max(0,min(r['start'] for r in rows)-32)
    hi=min(len(y),max(r['end'] for r in rows)+32)
    A=np.zeros((hi-lo,len(rows)),np.float64)
    for j,r in enumerate(rows): A[r['start']-lo:r['end']-lo,j]=r['wave']
    return lo,hi,A,np.asarray(y[lo:hi],np.float64)


def reconstruct(y,prepared,policy,cfg):
    """Deployable path: input mixture, template matches, frozen policy. NO references."""
    y=np.asarray(y,np.float64)
    if y.ndim!=1 or not len(y) or not np.isfinite(y).all(): raise ValueError('Invalid noisy waveform.')
    selected=select_evidence(prepared,float(policy['threshold']),cfg)
    z=y.copy(); log=[]; accepted=[]
    for rows in overlap_components(selected):
        lo,hi,A,v=component_matrix(y,rows)
        why=None; cond=None; disagreement=None; gains=None
        try:
            gains,cond=estimate_gains(A,v,[r['polarity'] for r in rows],policy['method'],cfg)
            if not np.isfinite(gains).all() or cond>cfg.max_condition: why='ill_conditioned_fit'
            if why is None and (np.any(np.abs(gains)<cfg.min_abs_gain) or np.any(np.abs(gains)>cfg.max_abs_gain)):
                why='gain_out_of_bounds'
            if why is None and any(float(g)*r['polarity']<=0 for g,r in zip(gains,rows)): why='gain_polarity_mismatch'
            if why is None:
                disagreement=gain_disagreement(A,v,gains,cfg)
                if disagreement>cfg.max_gain_disagreement: why='inconsistent_gain'
            noise=A@gains if gains is not None else np.zeros(len(v))
            removed_ratio=float(noise@noise)/max(float(v@v),1e-20)
            candidate=v-noise
            if why is None and removed_ratio>cfg.max_removal_energy_ratio: why='excessive_removal_energy'
            if why is None and float(candidate@candidate)>cfg.max_output_energy_ratio*max(float(v@v),1e-20): why='output_energy_increase'
            if why is None and (candidate.min()<-1. or candidate.max()>32767./32768.+1e-8): why='would_clip_pcm16'
        except (np.linalg.LinAlgError,FloatingPointError,ValueError) as e:
            why=f'numerical_fit_rejected:{type(e).__name__}'
        if why is None:
            # Reconstruct from ORIGINAL mixture, once. No generic mask or second fade.
            z[lo:hi]-=noise
            accepted.extend(dict(template_id=r['template_id'],start=r['start'],end=r['end'],gain=float(g),quality=r['quality'])
                            for g,r in zip(gains,rows))
        log.append(dict(template_ids=[r['template_id'] for r in rows],accepted=why is None,reason=why,
                        condition=cond,gain_disagreement=disagreement,
                        gains=None if gains is None else gains.tolist()))
    q=pcm_roundtrip(z); base=pcm_roundtrip(y)
    return q,dict(method=policy['method'],threshold=float(policy['threshold']),
                  proposed=len(prepared),selected=len(selected),accepted_templates=len(accepted),
                  changed=bool(np.any(q!=base)),change_db=residual_change_db(q,base),components=log,matches=accepted)


def reference_assisted_diagnostic(y,clean,prepared,policy,cfg):
    """DIAGNOSTIC ONLY. Same noisy-only matches, but clean is a nuisance regressor for ideal gains.
    Never exported as audio. Does NOT search using clean/GT and is NOT a deployable score.
    """
    y=np.asarray(y,np.float64); clean=np.asarray(clean,np.float64)
    z=y.copy(); nfit=0
    for rows in overlap_components(select_evidence(prepared,policy['threshold'],cfg)):
        lo,hi,A,v=component_matrix(y,rows)
        B=np.column_stack((A,clean[lo:hi]))
        try:
            g,_=fit_centered(B,v,ridge=1e-12)
            if np.isfinite(g).all(): z[lo:hi]-=A@g[:-1]; nfit+=len(rows)
        except np.linalg.LinAlgError: pass
    return dict(reference_assisted_sisdr_float=si_sdr(z,clean),
                reference_assisted_sisdr_pcm16=si_sdr(pcm_roundtrip(z),clean),fitted_templates=nfit)


TAG_RE=re.compile(r'</?[^<>]*>|\[[^\[\]]*\]')
def normalize_text(text):
    s=TAG_RE.sub(' ',str(text))
    s=''.join(' ' if unicodedata.category(c).startswith('P') else c for c in s)
    return ' '.join(s.lower().split())


def word_errors(reference,hypothesis):
    a=normalize_text(reference).split(); b=normalize_text(hypothesis).split()
    row=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        nxt=[i]
        for j,y in enumerate(b,1): nxt.append(min(nxt[-1]+1,row[j]+1,row[j-1]+(x!=y)))
        row=nxt
    return row[-1],len(a)


def pooled_wer(ids,refs,transcripts):
    errors=words=0
    for cid in ids:
        e,n=word_errors(refs[cid],transcripts[cid]['text']); errors+=e; words+=n
    if words==0: raise ValueError('No reference words: real WER cannot be evaluated.')
    return dict(wer=errors/words,errors=errors,words=words,clips=len(ids))


def hyp_text(h):
    if isinstance(h,str): return h.strip()
    if isinstance(h,dict) and isinstance(h.get('text'),str): return h['text'].strip()
    if hasattr(h,'text') and isinstance(h.text,str): return h.text.strip()
    if isinstance(h,(list,tuple)) and len(h)==1: return hyp_text(h[0])
    raise TypeError(f'Unrecognized ASR hypothesis type: {type(h)}. Refusing an invented empty transcript.')


class HashASR:
    """Model-revision + exact WAV byte hash cache. Never reuses an ID-only transcript."""
    def __init__(self,model,model_signature,cache_dir,batch_size=8):
        self.model=model; self.model_signature=model_signature
        self.cache_dir=Path(cache_dir); self.cache_dir.mkdir(parents=True,exist_ok=True)
        self.batch_size=batch_size
    def _call(self,paths):
        try:
            hs=self.model.transcribe([str(p) for p in paths],return_hypotheses=True)
            if isinstance(hs,(list,tuple)) and len(hs)==len(paths): return [hyp_text(h) for h in hs]
            if len(paths)==1: return [hyp_text(hs)]
        except Exception:
            if len(paths)==1: raise
        # A cardinality mismatch never silently truncates zip(...).
        return [self._call([p])[0] for p in paths]
    def transcribe(self,paths,progress=lambda x,**k:x):
        out={}; todo={}
        for cid,path in paths.items():
            sh=sha_file(path); key=sha_json(dict(model=self.model_signature,wav=sh))
            dest=self.cache_dir/(key+'.json')
            rec=None
            if dest.is_file():
                with open(dest,encoding='utf-8') as f: r=json.load(f)
                if r.get('model_signature')==self.model_signature and r.get('wav_sha256')==sh and isinstance(r.get('text'),str): rec=r
            if rec is not None: out[cid]=rec
            else:
                if key not in todo: todo[key]=dict(path=str(path),sha=sh,ids=[],dest=dest)
                todo[key]['ids'].append(cid)
        jobs=list(todo.values())
        print(f'ASR exact-hash hits: {len(out)} | unique WAVs to decode: {len(jobs)}')
        for a in progress(range(0,len(jobs),self.batch_size),desc='SraVaani final-waveform ASR'):
            ch=jobs[a:a+self.batch_size]; texts=self._call([j['path'] for j in ch])
            if len(texts)!=len(ch): raise RuntimeError('ASR cardinality mismatch.')
            for job,text in zip(ch,texts):
                # Empty output is allowed only when the real recognizer returned it.
                rec=dict(text=text,wav_sha256=job['sha'],model_signature=self.model_signature)
                put_json(job['dest'],rec)
                for cid in job['ids']: out[cid]=rec
        if set(out)!=set(paths): raise RuntimeError('Incomplete ASR coverage.')
        return out


def pack_submission(paths,asr,expected_frames,output_zip,sr=16000):
    """Only final audio and matching ASR enter the ZIP; diagnostics/clean refs never do."""
    ids=sorted(paths)
    if set(ids)!=set(asr) or set(ids)!=set(expected_frames): raise ValueError('Submission ID mismatch.')
    for cid in ids:
        if not cid or Path(cid).name!=cid or any(c in cid for c in ('/','\\','\n','\r')): raise ValueError('Unsafe clip ID.')
        h=wav_audit(paths[cid],expected_frames[cid],sr)
        if h!=asr[cid]['wav_sha256']: raise ValueError(f'ASR/audio hash mismatch: {cid}')
        if not isinstance(asr[cid]['text'],str): raise ValueError('ASR text must be a string.')
    p=Path(output_zip); p.parent.mkdir(parents=True,exist_ok=True); tmp=p.with_name(p.name+'.tmp')
    tr=''.join(json.dumps(dict(clip_id=c,text=asr[c]['text']),ensure_ascii=False,allow_nan=False)+'\n' for c in ids).encode('utf-8')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=9,allowZip64=True) as z:
        z.writestr('transcripts.jsonl',tr)
        for cid in ids: z.write(paths[cid],cid+'.wav')
    with zipfile.ZipFile(tmp) as z:
        wanted={'transcripts.jsonl'}|{c+'.wav' for c in ids}
        if len(z.namelist())!=len(wanted) or set(z.namelist())!=wanted: raise ValueError('Archive contents mismatch.')
        if z.testzip() is not None: raise ValueError('Archive CRC failure.')
        for cid in ids:
            data=z.read(cid+'.wav')
            if wav_audit(data,expected_frames[cid],sr)!=asr[cid]['wav_sha256']: raise ValueError('Packaged audio mismatch.')
        rows=[json.loads(x) for x in z.read('transcripts.jsonl').decode().splitlines()]
        if len(rows)!=len(ids) or {r['clip_id'] for r in rows}!=set(ids): raise ValueError('Packaged transcript mismatch.')
    os.replace(tmp,p)
    return dict(zip_path=str(p),zip_sha256=sha_file(p),clips=len(ids),transcripts_sha256=hashlib.sha256(tr).hexdigest())


In [ ]:
# Isolated mechanical tests; no released corpus or test labels used.
assert unit_tests()
_rng=np.random.default_rng(916)
_sr=16000; _N=32000; _tt=np.arange(_N)/_sr
_s=.025*np.sin(2*np.pi*163*_tt)
_t=(_rng.normal(size=6000)*np.hanning(6000)*.12).astype(np.float32)
_a=9231; _x=_s.copy(); _x[_a:_a+len(_t)]+=.70*_t
_cfg=ReconConfig()
_pp=[dict(template_id='fixture',onset=(_a+1)/_sr,offset=(_a+1+len(_t))/_sr,quality=.999,polarity=1)]
_prep=prepared_matches(_x,_pp,{'fixture':{'wave':_t}},_cfg)
assert _prep[0]['start']==_a
_z,_info=reconstruct(_x,_prep,dict(method='spectral_gls',threshold=.90),_cfg)
assert len(_z)==_N and _info['accepted_templates']==1
assert si_sdr(_z,_s)>si_sdr(pcm_roundtrip(_x),_s)+20
assert np.array_equal(_z[:_a],pcm_roundtrip(_x)[:_a])
_unchanged,_d=reconstruct(_x,[],dict(method='spectral_gls',threshold=.90),_cfg)
assert np.array_equal(_unchanged,pcm_roundtrip(_x)) and not _d['changed']
assert word_errors('one two three','one four three')==(1,3)
print('PASS: reconstruction, identity fallback, exact length, unchanged non-template samples, and word-error arithmetic.')
del _rng,_sr,_N,_tt,_s,_t,_a,_x,_cfg,_pp,_prep,_z,_info,_unchanged,_d


PASS: FFT timing, raw sample alignment, negative audio, self-retrieval guard, scorer, conservative edits.
PASS: reconstruction, identity fallback, exact length, unchanged non-template samples, and word-error arithmetic.


## 3. Frozen-policy preflight and cached released bank

In [ ]:
# 3. Load exact V5/V6 frozen selection artifacts and released validation references.
import json, pickle, zipfile, hashlib, time, shutil
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from IPython.display import display

print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| Torch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

torch.set_num_threads(min(4, os.cpu_count() or 1))
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False

required = [
    V5_DIR / 'frozen_bank.pkl',
    V5_DIR / 'frozen_split.json',
    V5_DIR / 'frozen_before_hold.json',
    V6_RUN / 'frozen_before_hold.json',
    VAL_CACHE,
    PHASE2_INPUT_SOURCE,
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

with open(V5_DIR/'frozen_bank.pkl', 'rb') as f:
    SELECTION_BANK = pickle.load(f)
with open(V5_DIR/'frozen_split.json', encoding='utf-8') as f:
    SPLIT = json.load(f)
with open(V5_DIR/'frozen_before_hold.json', encoding='utf-8') as f:
    V5_POLICY = json.load(f)
with open(V6_RUN/'frozen_before_hold.json', encoding='utf-8') as f:
    FROZEN_V6 = json.load(f)

if SELECTION_BANK.get('version') != VERSION:
    raise ValueError('Unexpected V5 bank version.')

MCFG = MatcherConfig(**SELECTION_BANK['cfg'])
SELECTION_MATCHER = ResidualMatcher(SELECTION_BANK, MCFG)

# Reproduce the exact selection provenance before deliberately refitting the deployment bank.
if SELECTION_MATCHER.signature != V5_POLICY['matcher_signature']:
    raise ValueError('V5 bank / V5 frozen policy mismatch.')
if digest(SPLIT) != V5_POLICY['split_sha']:
    raise ValueError('V5 split fingerprint mismatch.')

selection_bank_sha = sha_file(V5_DIR/'frozen_bank.pkl')
if FROZEN_V6.get('bank_sha256') != selection_bank_sha:
    raise ValueError('Frozen V6 policy was selected with a different V5 bank.')
if FROZEN_V6.get('matcher_signature') != SELECTION_MATCHER.signature:
    raise ValueError('Frozen V6 matcher signature mismatch.')

RCFG = ReconConfig(**FROZEN_V6['recon_config'])
FROZEN_POLICY = {
    'method': str(FROZEN_V6['method']),
    'threshold': float(FROZEN_V6['threshold']),
}

print('Frozen reconstruction policy:', FROZEN_POLICY)
print('Selection-bank templates:', len(SELECTION_BANK['records']))
print('Selection-bank SHA256:', selection_bank_sha)
print('No Phase-2 score is used to alter this method or threshold.')

# Load validation GT only to recover residual templates from released paired data.
with open(VAL_CACHE, 'rb') as f:
    raw = pickle.load(f)
records = raw if isinstance(raw, list) else [dict(v, clip_id=k) for k,v in raw.items()]

REFS, DOMAIN = {}, {}
for r in records:
    cid = str(r['clip_id'])
    REFS[cid] = event_list(r['gt'])
    DOMAIN[cid] = 'synthetic' if bool(r['synthetic']) else 'natural'
del raw, records

GROUPS = dict(SPLIT['group_key'])
for cid in REFS:
    GROUPS.setdefault(cid, source_group(cid))

BANK_IDS = list(SPLIT['bank'])
SELECT_SYN = list(SPLIT['select_syn'])
HOLD_SYN = list(SPLIT['hold_syn'])

ALL_RELEASED_SYN = sorted(set(BANK_IDS + SELECT_SYN + HOLD_SYN))
if not ALL_RELEASED_SYN:
    raise RuntimeError('No released synthetic clips found.')

if not all(DOMAIN.get(c) == 'synthetic' for c in ALL_RELEASED_SYN):
    raise ValueError('Deployment-bank source list contains a non-synthetic clip.')

# Locate released paired audio.
options = [Path(VAL_ROOT)] if VAL_ROOT is not None else VAL_ROOT_CANDIDATES
complete = [
    p for p in options
    if (p/'syntheticNoiseAudio').is_dir() and (p/'syntheticCleanRefAudio').is_dir()
]
if not complete:
    raise FileNotFoundError(
        'Set VAL_ROOT to the released validation folder containing '
        'syntheticNoiseAudio/ and syntheticCleanRefAudio/.'
    )
VAL_ROOT = complete[0]

def index_exact(root, required):
    idx = {}
    for p in Path(root).rglob('*.wav'):
        cid = p.stem
        if cid in required:
            if cid in idx:
                raise ValueError(f'Duplicate WAV stem {cid} under {root}')
            idx[cid] = str(p)
    miss = set(required) - set(idx)
    if miss:
        raise FileNotFoundError(
            f'{root}: missing {len(miss)} required files; examples {sorted(miss)[:3]}'
        )
    return idx

SYN_NOISY_PATHS = index_exact(VAL_ROOT/'syntheticNoiseAudio', ALL_RELEASED_SYN)
SYN_CLEAN_PATHS = index_exact(VAL_ROOT/'syntheticCleanRefAudio', ALL_RELEASED_SYN)

print('Released paired synthetic clips for final refit:', len(ALL_RELEASED_SYN))
print('Validation root:', VAL_ROOT)


Python: 3.13.15 | NumPy: 2.1.3 | Torch: 2.11.0+cu130
Device: Tesla T4
Frozen reconstruction policy: {'method': 'block_irls', 'threshold': 0.55}
Selection-bank templates: 1349
Selection-bank SHA256: 50d6991ea6daf8eac317a8f57358706250b1f23a56244d11e0289328cb6f1536
No Phase-2 score is used to alter this method or threshold.
Released paired synthetic clips for final refit: 695
Validation root: /content/drive/MyDrive/PHD/Datathon/validation


In [ ]:
# 4. FINAL DEPLOYMENT REFIT: all released paired synthetic data, frozen extraction recipe.
#
# IMPORTANT:
# - No threshold/method is retuned here.
# - No Phase-2 WAV contributes a template.
# - Exact waveform deduplication and the original V5 max-template cap remain unchanged.

DEPLOY_BANK_PATH = OUT_ROOT / 'full_released_deployment_bank.pkl'

file_identity = [
    (
        cid,
        os.path.getsize(SYN_NOISY_PATHS[cid]),
        os.path.getmtime(SYN_NOISY_PATHS[cid]),
        os.path.getsize(SYN_CLEAN_PATHS[cid]),
        os.path.getmtime(SYN_CLEAN_PATHS[cid]),
    )
    for cid in ALL_RELEASED_SYN
]

DEPLOY_INPUT_KEY = digest({
    'version': 'phase2_v24_full_released_refit',
    'matcher_config': asdict(MCFG),
    'frozen_policy': FROZEN_POLICY,
    'released_ids': ALL_RELEASED_SYN,
    'files': file_identity,
})

DEPLOY_BANK = None
if DEPLOY_BANK_PATH.is_file():
    with open(DEPLOY_BANK_PATH, 'rb') as f:
        candidate = pickle.load(f)
    if candidate.get('deployment_input_key') == DEPLOY_INPUT_KEY:
        DEPLOY_BANK = candidate
        print('Loaded cached deployment bank.')

if DEPLOY_BANK is None:
    DEPLOY_BANK = build_bank(
        ALL_RELEASED_SYN,
        REFS,
        SYN_NOISY_PATHS,
        SYN_CLEAN_PATHS,
        GROUPS,
        MCFG,
        progress=tqdm,
    )
    DEPLOY_BANK['deployment_input_key'] = DEPLOY_INPUT_KEY
    DEPLOY_BANK['deployment_provenance'] = {
        'selection_bank_sha256': selection_bank_sha,
        'selection_v6_run_key': V6_RUN_KEY,
        'all_released_paired_synthetic': True,
        'phase2_data_used_for_bank': False,
        'policy_retuned_after_refit': False,
    }
    atomic_pickle(DEPLOY_BANK_PATH, DEPLOY_BANK)
    print('Saved deployment bank:', DEPLOY_BANK_PATH)

DEPLOY_MATCHER = ResidualMatcher(DEPLOY_BANK, MCFG)
DEPLOY_BANK_BY_ID = {r['tid']: r for r in DEPLOY_BANK['records']}

if len(DEPLOY_BANK_BY_ID) != len(DEPLOY_BANK['records']):
    raise ValueError('Duplicate template ID in deployment bank.')

DEPLOY_BANK_SHA = sha_file(DEPLOY_BANK_PATH)
DEPLOY_KEY = sha_json({
    'selection_v6_run': V6_RUN_KEY,
    'selection_policy': FROZEN_POLICY,
    'deployment_matcher': DEPLOY_MATCHER.signature,
    'deployment_bank_sha256': DEPLOY_BANK_SHA,
})[:16]

RUN_DIR = OUT_ROOT / ('run_' + DEPLOY_KEY)
LOCAL = LOCAL_ROOT / ('run_' + DEPLOY_KEY)
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)

put_json(RUN_DIR/'deployment_provenance.json', {
    'version': 'track2_phase2_v24_full_released_refit',
    'deploy_key': DEPLOY_KEY,
    'selection_v6_run_key': V6_RUN_KEY,
    'selection_bank_sha256': selection_bank_sha,
    'selection_matcher_signature': SELECTION_MATCHER.signature,
    'frozen_policy': FROZEN_POLICY,
    'deployment_bank_sha256': DEPLOY_BANK_SHA,
    'deployment_matcher_signature': DEPLOY_MATCHER.signature,
    'released_paired_synthetic_clips': len(ALL_RELEASED_SYN),
    'templates': len(DEPLOY_BANK['records']),
    'exact_duplicates_removed': int(DEPLOY_BANK.get('deduplicated', 0)),
    'templates_omitted_by_original_cap': int(DEPLOY_BANK.get('capped', 0)),
    'phase2_labels_or_clean_used': False,
})

print('='*78)
print('FULL RELEASED DEPLOYMENT BANK READY')
print('Released paired clips        :', len(ALL_RELEASED_SYN))
print('Templates                    :', len(DEPLOY_BANK['records']))
print('Exact duplicates removed     :', DEPLOY_BANK.get('deduplicated', 0))
print('Omitted by original V5 cap   :', DEPLOY_BANK.get('capped', 0))
print('Frozen method                :', FROZEN_POLICY['method'])
print('Frozen threshold             :', FROZEN_POLICY['threshold'])
print('Deployment matcher signature :', DEPLOY_MATCHER.signature)
print('Run directory                :', RUN_DIR)
print('='*78)

if int(DEPLOY_BANK.get('capped', 0)) > 0:
    print(
        'NOTE: the original V5 max-template cap is active. '
        'This notebook intentionally does not enlarge it after model selection.'
    )


Loaded cached deployment bank.
FULL RELEASED DEPLOYMENT BANK READY
Released paired clips        : 695
Templates                    : 2000
Exact duplicates removed     : 0
Omitted by original V5 cap   : 207
Frozen method                : block_irls
Frozen threshold             : 0.55
Deployment matcher signature : 3311acc7bacbee56c51005bfc357069bc09f2d90611f559067359fa738ae0c03
Run directory                : /content/drive/MyDrive/Datathon/track2_phase2_v24_fullbank/run_6ede50137c77f307
NOTE: the original V5 max-template cap is active. This notebook intentionally does not enlarge it after model selection.


## 4. Pin input and create dictionary-all outputs
The original V24 score threshold and reconstruction safeguards are retained. No forced subtraction of weak matches is added. Unchanged clips are counted and disclosed. Output WAVs are saved persistently to resume after disconnects.

In [ ]:
# 6. Discover Phase-2 IDs directly from the NEW Codabench input.
def discover_wav_ids(path):
    p = Path(path)
    names = []
    if p.is_file() and zipfile.is_zipfile(p):
        with zipfile.ZipFile(p) as z:
            names = [
                x.filename for x in z.infolist()
                if (not x.is_dir())
                and x.filename.lower().endswith('.wav')
                and '__MACOSX' not in x.filename
            ]
    elif p.is_dir():
        names = [str(q) for q in p.rglob('*.wav') if q.is_file()]
    else:
        raise FileNotFoundError(p)

    ids = [Path(n).stem for n in names]
    if len(ids) != len(set(ids)):
        from collections import Counter
        dups = [k for k,v in Counter(ids).items() if v > 1]
        raise ValueError(f'Duplicate Phase-2 WAV stems: {dups[:5]}')
    return sorted(ids)

TEST_IDS = discover_wav_ids(PHASE2_INPUT_SOURCE)

if len(TEST_IDS) != EXPECTED_PHASE2_CLIPS:
    raise ValueError(
        f'Expected {EXPECTED_PHASE2_CLIPS} Phase-2 WAVs, found {len(TEST_IDS)}. '
        'Check that this is the NEW Final Submission (Phase-2) input_data.'
    )

if set(TEST_IDS) & set(REFS):
    raise ValueError('Phase-2 IDs overlap labelled validation IDs.')

TEST_SOURCE = StrictWaveSource(PHASE2_INPUT_SOURCE, TEST_IDS)
TEST_MANIFEST_SHA = sha_json(TEST_IDS)

if PHASE2_INPUT_SOURCE.is_file():
    TEST_SOURCE_SHA = sha_file(PHASE2_INPUT_SOURCE)
else:
    TEST_SOURCE_SHA = None

put_json(RUN_DIR/'phase2_input_identity.json', {
    'source': str(PHASE2_INPUT_SOURCE),
    'source_sha256': TEST_SOURCE_SHA,
    'ids_sha256': TEST_MANIFEST_SHA,
    'clips': len(TEST_IDS),
    'expected_clips': EXPECTED_PHASE2_CLIPS,
})

print('PHASE-2 INPUT VERIFIED')
print('Clips       :', len(TEST_IDS))
print('IDs SHA256  :', TEST_MANIFEST_SHA)
print('Input SHA256:', TEST_SOURCE_SHA)
print('First IDs   :', TEST_IDS[:3])


PHASE-2 INPUT VERIFIED
Clips       : 3178
IDs SHA256  : 18c8951e98da87d5d50c4f9a9eb4d75dcfcdc9d4a01df068ada7981ac4304c1a
Input SHA256: 022d36ba6fcf1e40a850b96fe17ae329808025b894fbe950d73362d693ee425a
First IDs   : ['indoMLDatathon_202608_00054484', 'indoMLDatathon_202608_00076758', 'indoMLDatathon_202608_00090721']


In [ ]:
ORIGINAL_V24_RUN=RUN_DIR
if 'asr_model' in globals():asr_model.to('cpu')
if 'enh_model' in globals():enh_model.to('cpu')
gc.collect();torch.cuda.empty_cache()
# 7. Phase-2 reconstruction using the FULL released deployment bank.
# Every decision is clip-local. No test-to-test adaptation or template harvesting.

FINAL_PATHS = {}
EXPECTED_FRAMES = {}
TEST_AUDIT = []

LOCAL_WAVS = RUN_DIR/'persistent_final_wavs'
RECORDS = RUN_DIR/'test_records'
LOCAL_WAVS.mkdir(parents=True, exist_ok=True)
RECORDS.mkdir(parents=True, exist_ok=True)

POLICY_SHA = sha_json({
    'frozen_selection_policy': FROZEN_POLICY,
    'deployment_matcher_signature': DEPLOY_MATCHER.signature,
    'deployment_bank_sha256': DEPLOY_BANK_SHA,
})

def get_phase2_proposals(cid, y):
    if wave_sha(y) in DEPLOY_MATCHER.source_hashes:
        raise ValueError(f'Phase-2 waveform duplicates a released bank source: {cid}')
    p = RUN_DIR/'phase2_queries'/(cid+'.json')
    return persist_query(DEPLOY_MATCHER, y, p, exclude_group=None)

start_time = time.time()

for j, cid in enumerate(tqdm(TEST_IDS, desc='V24 full-bank Phase-2 reconstruction')):
    y = TEST_SOURCE.load(cid)
    source_sha = wave_sha(y)
    n = len(y)
    EXPECTED_FRAMES[cid] = n

    key = sha_json({
        'policy': POLICY_SHA,
        'wave': source_sha,
        'cid': cid,
    })

    out_path = LOCAL_WAVS/(cid+'.wav')
    rec_path = RECORDS/(cid+'.json')
    rec = None

    if out_path.is_file() and rec_path.is_file():
        with open(rec_path, encoding='utf-8') as f:
            candidate = json.load(f)
        if candidate.get('key') == key:
            if wav_audit(out_path, n) == candidate.get('final_wav_sha256'):
                rec = candidate

    if rec is None:
        props = get_phase2_proposals(cid, y)
        prepared = prepared_matches(y, props, DEPLOY_BANK_BY_ID, RCFG)
        z, info = reconstruct(y, prepared, FROZEN_POLICY, RCFG)

        out_sha = write_pcm(out_path, z)
        wav_audit(out_path, n)

        rec = {
            'key': key,
            'clip_id': cid,
            'source_wave_sha256': source_sha,
            'expected_frames': n,
            'final_wav_sha256': out_sha,
            'policy_sha256': POLICY_SHA,
            'proposal_count': len(props),
            'details': info,
        }
        put_json(rec_path, rec)

    FINAL_PATHS[cid] = str(out_path)
    info = rec['details']

    TEST_AUDIT.append({
        'clip_id': cid,
        'proposal_count': int(rec.get('proposal_count', info.get('proposed', 0))),
        'selected': int(info.get('selected', 0)),
        'accepted_templates': int(info.get('accepted_templates', 0)),
        'changed': bool(info.get('changed', False)),
        'change_db': float(info.get('change_db', 0.0)),
        'source_wave_sha256': source_sha,
        'final_wav_sha256': rec['final_wav_sha256'],
    })

    if j == 9:
        print(f'First 10 clips took {time.time()-start_time:.1f}s')
    if (j + 1) % 100 == 0:
        recent = TEST_AUDIT[-100:]
        print(
            f'Processed {j+1}/{len(TEST_IDS)} | '
            f'changed total={sum(r["changed"] for r in TEST_AUDIT)} | '
            f'accepted events total={sum(r["accepted_templates"] for r in TEST_AUDIT)} | '
            f'changed last100={sum(r["changed"] for r in recent)}'
        )

if set(FINAL_PATHS) != set(TEST_IDS):
    raise ValueError('Incomplete Phase-2 output coverage.')

TEST_AUDIT_DF = pd.DataFrame(TEST_AUDIT)
TEST_AUDIT_DF.to_csv(RUN_DIR/'phase2_reconstruction_audit.csv', index=False)

changed = int(TEST_AUDIT_DF.changed.sum())
accepted = int(TEST_AUDIT_DF.accepted_templates.sum())
proposals = int(TEST_AUDIT_DF.proposal_count.sum())

print('\n' + '='*78)
print('PHASE-2 WAVEFORM PASS COMPLETE')
print('Clips                 :', len(TEST_IDS))
print('Clips changed         :', changed)
print('Changed fraction      :', round(changed/len(TEST_IDS), 4))
print('Matcher proposals     :', proposals)
print('Accepted templates    :', accepted)
print('Mean proposals/clip   :', round(proposals/len(TEST_IDS), 3))
print('Mean accepted/clip    :', round(accepted/len(TEST_IDS), 3))
print('='*78)

if changed == 0:
    raise RuntimeError(
        'Full released bank changed zero clips. Stop before ASR/submission; '
        'this candidate has no reason to beat identity.'
    )

DICT_PATHS=dict(FINAL_PATHS)
DICT_AUDIT={r['clip_id']:r for r in TEST_AUDIT}
DICT_UNCHANGED=[cid for cid in TEST_IDS if not DICT_AUDIT[cid]['changed']]
print('Dictionary searched ALL:',len(TEST_IDS))
print('Dictionary actually changed:',len(TEST_IDS)-len(DICT_UNCHANGED))
print('Unmatched/unchanged inputs:',len(DICT_UNCHANGED))


V24 full-bank Phase-2 reconstruction:   0%|          | 0/3178 [00:00<?, ?it/s]

First 10 clips took 6.0s
Processed 100/3178 | changed total=58 | accepted events total=281 | changed last100=58
Processed 200/3178 | changed total=109 | accepted events total=517 | changed last100=51
Processed 300/3178 | changed total=163 | accepted events total=789 | changed last100=54
Processed 400/3178 | changed total=224 | accepted events total=1087 | changed last100=61
Processed 500/3178 | changed total=276 | accepted events total=1331 | changed last100=52
Processed 600/3178 | changed total=334 | accepted events total=1606 | changed last100=58
Processed 700/3178 | changed total=386 | accepted events total=1846 | changed last100=52
Processed 800/3178 | changed total=451 | accepted events total=2151 | changed last100=65
Processed 900/3178 | changed total=505 | accepted events total=2407 | changed last100=54
Processed 1000/3178 | changed total=554 | accepted events total=2630 | changed last100=49
Processed 1100/3178 | changed total=604 | accepted events total=2863 | changed last100=5

## 5. Neural fallback on dictionary-unchanged clips
“Dictionary-unchanged” is a match outcome, not a verified natural label. Unmatched synthetic clips also receive the neural model; naturally recorded clips with accepted dictionary subtraction retain that dictionary result. Neural inference uses the V36 dns64 residual-adaptive architecture, 51,200-sample chunks, 25,600-sample hop, Hann overlap-add and sign averaging at 16 kHz. No extra peak normalization or masking is introduced.

In [ ]:
# Exact V36 noise-adaptive Demucs architecture and inference settings.
DEPLOY_MATCHER.cache.clear();DEPLOY_MATCHER.cache_bytes=0
gc.collect();torch.cuda.empty_cache()
if importlib.util.find_spec('denoiser') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','denoiser==0.1.5'])
from torch import nn
from denoiser import pretrained as demucs_pretrained
from denoiser.resample import upsample2,downsample2
NEURAL_DEVICE=torch.device('cuda')
DEMUCS_CHUNK=51200;DEMUCS_HOP=25600
class SpectralNoiseConditioner(nn.Module):
    def __init__(self,n_fft=512,hop=160,embedding=128):
        super().__init__();self.n_fft=n_fft;self.hop=hop
        self.register_buffer("window",torch.hann_window(n_fft),persistent=False)
        features=2*(n_fft//2+1)
        self.net=nn.Sequential(nn.LayerNorm(features),nn.Linear(features,256),nn.SiLU(),
                               nn.Linear(256,embedding),nn.SiLU())
    def forward(self,wave):
        with torch.amp.autocast(device_type=wave.device.type,enabled=False):
            spec=torch.stft(wave.float(),self.n_fft,hop_length=self.hop,window=self.window.float(),
                            return_complex=True,pad_mode="constant").abs()
            feat=torch.log1p(10*spec);feat=torch.cat([feat.mean(-1),feat.std(-1)],-1)
        return self.net(feat)

class NoiseAdaptiveResidualDemucs(nn.Module):
    """Exact Demucs forward plus continuous FiLM and learned residual strength."""
    def __init__(self,demucs,embedding=128,film_scale=.10,strength_range=.15):
        super().__init__();self.base=demucs;self.conditioner=SpectralNoiseConditioner(embedding=embedding)
        bottleneck=int(self.base.encoder[-1][0].out_channels)
        self.film=nn.Linear(embedding,2*bottleneck)
        self.strength=nn.Linear(embedding,1)
        nn.init.zeros_(self.film.weight);nn.init.zeros_(self.film.bias)
        nn.init.zeros_(self.strength.weight);nn.init.zeros_(self.strength.bias)
        self.film_scale=float(film_scale);self.strength_range=float(strength_range)
    def forward(self,mix,return_aux=False):
        if mix.dim()==2:mix=mix.unsqueeze(1)
        original=mix;embedding=self.conditioner(original[:,0])
        if self.base.normalize:
            std=original.mean(1,keepdim=True).std(-1,keepdim=True);x=original/(self.base.floor+std)
        else:std=1.;x=original
        length=x.shape[-1];x=F.pad(x,(0,self.base.valid_length(length)-length))
        if self.base.resample==2:x=upsample2(x)
        elif self.base.resample==4:x=upsample2(upsample2(x))
        skips=[]
        for encode in self.base.encoder:x=encode(x);skips.append(x)
        x=x.permute(2,0,1);x,_=self.base.lstm(x);x=x.permute(1,2,0)
        gamma,beta=self.film(embedding).chunk(2,-1)
        x=x*(1+self.film_scale*torch.tanh(gamma)[:,:,None])+self.film_scale*torch.tanh(beta)[:,:,None]
        for decode in self.base.decoder:
            skip=skips.pop(-1);x=decode(x+skip[...,:x.shape[-1]])
        if self.base.resample==2:x=downsample2(x)
        elif self.base.resample==4:x=downsample2(downsample2(x))
        speech_candidate=std*x[...,:length]
        strength=1+self.strength_range*torch.tanh(self.strength(embedding))[:,:,None]
        # Stable algebra: when strength==1 this is exactly speech_candidate.
        residual=original-speech_candidate
        predicted_noise=strength*residual
        enhanced=speech_candidate+(1-strength)*residual
        return (enhanced,predicted_noise,strength,embedding) if return_aux else enhanced
demucs_model=NoiseAdaptiveResidualDemucs(demucs_pretrained.dns64(pretrained=False))
if demucs_model.base.sample_rate!=16000 or demucs_model.base.chin!=1:
    raise RuntimeError('Unexpected dns64 sample rate/channels')
state=torch.load(NEURAL_CHECKPOINT,map_location='cpu',weights_only=False)
demucs_model.load_state_dict(state.get('model',state),strict=True)
del state
demucs_model.to(NEURAL_DEVICE).float().eval()
for module in demucs_model.modules():
    if isinstance(module,nn.RNNBase):module.flatten_parameters()
NEURAL_SHA=sha_file(NEURAL_CHECKPOINT)
LAST_MODEL_CALLS=0
@torch.inference_mode()
def demucs_fullstrength(y):
    global LAST_MODEL_CALLS
    y=np.asarray(y,np.float32);out=np.zeros(len(y),np.float64);den=np.zeros(len(y),np.float64)
    weight=np.maximum(np.hanning(DEMUCS_CHUNK),1e-3);LAST_MODEL_CALLS=0
    for start in range(0,len(y),DEMUCS_HOP):
        count=min(DEMUCS_CHUNK,len(y)-start);seg=np.pad(y[start:start+count],(0,DEMUCS_CHUNK-count)).astype(np.float32)
        xt=torch.from_numpy(seg)[None].to(NEURAL_DEVICE)
        _,n1,_,_=demucs_model(xt[:,None],return_aux=True)
        if NEURAL_SIGN_ENSEMBLE:
            _,n2,_,_=demucs_model(-xt[:,None],return_aux=True);pred_n=.5*(n1-n2);LAST_MODEL_CALLS+=2
        else:pred_n=n1;LAST_MODEL_CALLS+=1
        z=(xt[:,None]-1.0*pred_n)[0,0].float().cpu().numpy()
        if z.shape!=(DEMUCS_CHUNK,) or not np.isfinite(z).all():raise RuntimeError("Invalid enhancement; no fallback")
        out[start:start+count]+=z[:count]*weight[:count];den[start:start+count]+=weight[:count]
        if start+DEMUCS_CHUNK>=len(y):break
    if (den<=0).any():raise RuntimeError("Uncovered samples")
    return (out/den).astype(np.float32)

def neural_once(x):
    return demucs_fullstrength(x)
HYBRID_CONFIG=dict(dictionary_policy=POLICY_SHA,neural_tag=NEURAL_TAG,
    checkpoint_sha256=NEURAL_SHA,alpha=NEURAL_ALPHA,sr=16000,
    segment=NEURAL_SEGMENT_SECONDS,hop=NEURAL_HOP_SECONDS,
    sign_ensemble=NEURAL_SIGN_ENSEMBLE,routing='dictionary_pcm_changed_else_neural_v1')
HYBRID_KEY=sha_json(HYBRID_CONFIG)[:16]
HYBRID_ROOT=EXPERIMENT_ROOT/('hybrid_'+HYBRID_KEY)
HYBRID_ROOT.mkdir(parents=True,exist_ok=True)
HYBRID_PATHS={};HYBRID_AUDIT=[]

for cid in tqdm(TEST_IDS,desc='Dictionary matched / neural unmatched'):
    if DICT_AUDIT[cid]['changed']:
        HYBRID_PATHS[cid]=DICT_PATHS[cid]
        HYBRID_AUDIT.append(dict(clip_id=cid,route='dictionary',changed=True))
        continue
    x=TEST_SOURCE.load(cid)
    p=HYBRID_ROOT/(cid+'.wav');ap=HYBRID_ROOT/(cid+'.audit.json')
    key=sha_json(dict(config=HYBRID_CONFIG,input=wave_sha(x)))
    rec=None
    if p.is_file() and ap.is_file():
        trial=json.loads(ap.read_text())
        if trial.get('key')==key and trial.get('wav_sha256')==wav_audit(p,len(x)):
            rec=trial
    if rec is None:
        y=neural_once(x)
        z=(x-float(NEURAL_ALPHA)*(x-y)).astype(np.float32)
        write_pcm(p,z)
        exported=read_wave_strict(p)
        if not np.any(exported):raise RuntimeError(f'{cid}: zero neural output')
        rec=dict(clip_id=cid,key=key,route='neural',wav_sha256=wav_audit(p,len(x)),
            changed=bool(np.any(canonical_pcm(z)!=canonical_pcm(x))),
            clipped_samples=int(np.count_nonzero((z<-1)|(z>32767/32768))))
        put_json(ap,rec)
    HYBRID_PATHS[cid]=str(p);HYBRID_AUDIT.append(rec)
pd.DataFrame(HYBRID_AUDIT).to_csv(HYBRID_ROOT/'routing_audit.csv',index=False)
print(pd.DataFrame(HYBRID_AUDIT).groupby('route').agg(clips=('clip_id','size'),changed=('changed','sum')))
unchanged=[r['clip_id'] for r in HYBRID_AUDIT if not r['changed']]
print('Hybrid unchanged outputs:',len(unchanged))
if unchanged:raise RuntimeError('Inspect unchanged neural outputs; do not add artificial changes.')
put_json(HYBRID_ROOT/'inference_config.json',HYBRID_CONFIG)
demucs_model.to('cpu');del demucs_model
gc.collect();torch.cuda.empty_cache()


Dictionary matched / neural unmatched:   0%|          | 0/3178 [00:00<?, ?it/s]

            clips  changed
route                     
dictionary   1716     1716
neural       1462     1462
Hybrid unchanged outputs: 0


## 6. Load pinned SraVaani and use one-file, content-hashed ASR
The audited single-file call runs with gradients disabled but outside inference mode to avoid the prior TorchScript version-counter error. Submitted texts come from the final WAVs, never source/reference transcripts.

In [ ]:
if 'asr_model' in globals():
    asr_model.to('cpu');del asr_model
if 'ASR' in globals():
    del ASR
gc.collect();torch.cuda.empty_cache()
# 5. Load the exact pinned SraVaani revision used by the Phase-1 reconstruction system.
ASR_ENGINE = None
ASR_MODEL_SIGNATURE = None

if RUN_ASR:
    try:
        from transformers import AutoModel
    except ImportError as e:
        raise RuntimeError(
            'transformers is missing. Use the same working Colab environment '
            'that successfully ran the Phase-2 V23 package.'
        ) from e

    token = os.environ.get('HF_TOKEN')
    if token is None:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None

    if not OLD_ASR_PIN.is_file():
        raise FileNotFoundError(
            f'Expected exact V6 SraVaani revision pin at {OLD_ASR_PIN}. '
            'Do not silently switch ASR revisions.'
        )

    with open(OLD_ASR_PIN, encoding='utf-8') as f:
        pin = json.load(f)

    if pin.get('repo') != ASR_REPO:
        raise ValueError('SraVaani repository mismatch.')

    resolved_revision = str(pin['revision'])
    if len(resolved_revision) != 40:
        raise ValueError('Expected immutable 40-character SraVaani commit SHA.')

    ASR_MODEL_SIGNATURE = sha_json({
        'repo': ASR_REPO,
        'revision': resolved_revision,
        'task': 'transcribe_return_hypotheses',
    })

    put_json(RUN_DIR/'asr_identity.json', {
        'repo': ASR_REPO,
        'revision': resolved_revision,
        'signature': ASR_MODEL_SIGNATURE,
    })

    # Release matcher FFT cache before loading ASR.
    DEPLOY_MATCHER.cache.clear()
    DEPLOY_MATCHER.cache_bytes = 0
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print('Loading pinned SraVaani:', resolved_revision)
    asr_model = AutoModel.from_pretrained(
        ASR_REPO,
        revision=resolved_revision,
        code_revision=resolved_revision,
        trust_remote_code=True,
        token=token,
    ).to(DEPLOY_MATCHER.device).eval()

    ASR_ENGINE = HashASR(
        asr_model,
        ASR_MODEL_SIGNATURE,
        OUT_ROOT/'asr_cache_by_content',
        ASR_BATCH_SIZE,
    )
    print('SraVaani ready.')
else:
    print('RUN_ASR=False: waveform research only; final ZIP will not be created.')

# Override the original batched helper with a single-file hash-staged call.
ASR_ENGINE.batch_size=1
ASR_STAGE=LOCAL_ROOT/'final_two_asr_inputs';ASR_STAGE.mkdir(parents=True,exist_ok=True)
def exact_single_call(paths):
    texts=[]
    for path in paths:
        h=sha_file(path);staged=ASR_STAGE/(h+'.wav')
        if not staged.is_file() or sha_file(staged)!=h:shutil.copyfile(path,staged)
        if sha_file(staged)!=h:raise IOError('ASR staging checksum mismatch')
        with torch.inference_mode(False),torch.no_grad():
            hyp=asr_model.transcribe([str(staged)],return_hypotheses=True)
        texts.append(hyp_text(hyp))
    return texts
ASR_ENGINE._call=exact_single_call


## 7. Package both experiments and verify archive contents
A may contain unchanged clips and remains a dictionary system. B is explicitly match-routed. Both retain the rule-based method risks stated above. The audit below checks packaging and ASR/audio hash agreement; it cannot establish method eligibility or predict the hidden score.

In [ ]:
VARIANTS=[
    ('DICTIONARY_ALL_QUERIED',DICT_PATHS,False),
    ('DICTIONARY_MATCHED_NEURAL_UNMATCHED',HYBRID_PATHS,True),
]
PACKAGES=[]
for label,paths,is_hybrid in VARIANTS:
    if set(paths)!=set(TEST_IDS):raise RuntimeError('Incomplete output IDs')
    final_asr=ASR_ENGINE.transcribe(paths,progress=tqdm)
    key=HYBRID_KEY if is_hybrid else DEPLOY_KEY
    dest=EXPERIMENT_ROOT/f'submission_Track2_PHASE2_FINAL_{label}_{key}.zip'
    local_zip=LOCAL_ROOT/dest.name
    package=pack_submission(paths,final_asr,EXPECTED_FRAMES,local_zip)
    tmp=dest.with_name(dest.name+'.tmp')
    shutil.copyfile(local_zip,tmp)
    if sha_file(tmp)!=package['zip_sha256']:raise IOError('Drive copy checksum mismatch')
    os.replace(tmp,dest)
    with zipfile.ZipFile(dest) as z:
        if z.testzip() is not None:raise RuntimeError('Final ZIP CRC failure')
        wanted={cid+'.wav' for cid in TEST_IDS}|{'transcripts.jsonl'}
        if set(z.namelist())!=wanted or len(z.namelist())!=len(wanted):
            raise RuntimeError('Final archive layout mismatch')
    manifest=dict(variant=label,zip_path=str(dest),package=package,
        dictionary_lookup_at_inference=True,
        match_routing_at_inference=is_hybrid,
        synthetic_natural_ground_truth_used=False,
        dictionary_queried_clips=len(TEST_IDS),
        dictionary_changed_clips=len(TEST_IDS)-len(DICT_UNCHANGED),
        unchanged_clips=0 if is_hybrid else len(DICT_UNCHANGED),
        neural_config=HYBRID_CONFIG if is_hybrid else None,
        method_eligibility='not_established_dictionary_guidance_applies')
    put_json(EXPERIMENT_ROOT/(dest.stem+'_manifest.json'),manifest)
    PACKAGES.append(manifest)
    print('READY:',dest)
TEST_SOURCE.close()
put_json(EXPERIMENT_ROOT/'two_submissions.json',PACKAGES)
display(pd.DataFrame([dict(variant=m['variant'],zip_path=m['zip_path'],
    unchanged=m['unchanged_clips'],sha256=m['package']['zip_sha256']) for m in PACKAGES]))
